# Calculate GRUs parameters from available dataset

Function to Calculate zonal statistics using raster zone input

In [3]:
# importing the necessary libraries
import os
import glob
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.windows import Window
from rasterio.vrt import WarpedVRT
from collections import defaultdict
import pandas as pd
from natsort import natsorted
import xarray as xr
from scipy import ndimage
from concurrent.futures import ThreadPoolExecutor

def calculate_zonal_stats_streamed(
    zone_raster_path,
    value_raster_path,
    bands=None,
    tile_size=1024,
    output_csv_path=None,
    per_band_output_folder=None,
    resampling_method=Resampling.nearest,
    exclude_values=None,
    value_transform=None,
    include_mean=True,
    include_median=False,
    include_std=False,
    include_sum=True,
    include_min=True,
    include_max=True,
    include_count=True,
    parallel=False,
    max_workers=4
):
    exclude_values = list(exclude_values) if exclude_values else []
    paths = (
        natsorted(glob.glob(value_raster_path))
        if isinstance(value_raster_path, str) and "*" in value_raster_path
        else [value_raster_path] if isinstance(value_raster_path, (str, os.PathLike))
        else value_raster_path
    )

    all_dfs = []

    with rasterio.open(zone_raster_path) as zone_src:
        zone_crs, zone_nod = zone_src.crs, zone_src.nodata
        zone_w, zone_h = zone_src.width, zone_src.height

        def process_path(val_path):
            base = os.path.splitext(os.path.basename(val_path))[0]
            print(f"📊 Processing {base}")

            if val_path.lower().endswith(".nc"):
                with xr.open_dataset(val_path) as ds:
                    time_var = ds.get('time', None)
                    time_indices = (
                        np.argsort(time_var.values) if time_var is not None and time_var.size > 0
                        else range(len(ds.get('time', [])) or 1)
                    )
                    for var in sorted(ds.data_vars):
                        arr = ds[var].values
                        has_time = arr.ndim == 3
                        times = time_indices if has_time else [None]
                        for t_idx, t in enumerate(times):
                            data = arr[t] if has_time else arr
                            time_label = f"_t{t_idx+1}" if has_time else ""
                            label = f"{base}_{var}{time_label}"
                            stats = _run_array_stats(data, zone_src, tile_size,
                                                     exclude_values, value_transform,
                                                     include_count, include_sum, include_mean,
                                                     include_min, include_max, include_median, include_std,
                                                     zone_nod)
                            df = _build_stats_df(stats, label,
                                                 include_count, include_sum, include_mean,
                                                 include_min, include_max, include_median, include_std)
                            all_dfs.append(df)
                            _save_per_band(df, per_band_output_folder, label)

            else:
                with rasterio.open(val_path) as src:
                    src_nod = src.nodata
                    tgt_bands = bands or list(range(1, src.count + 1))
                    for b in sorted(tgt_bands):
                        try:
                            desc = src.descriptions[b - 1]
                            bn = desc if desc and desc.strip() else f"band{b}"
                        except (IndexError, TypeError):
                            bn = f"band{b}"
                        label = f"{base}_{bn}"
                        print(f"🔍 Processing band {b}: {label}")

                        with WarpedVRT(src, crs=zone_crs,
                                       transform=zone_src.transform,
                                       width=zone_w, height=zone_h,
                                       resampling=resampling_method) as vrt:
                            stats = _run_raster_stats_fast(
                                vrt, b, zone_src, tile_size,
                                zone_nod, src_nod,
                                exclude_values, value_transform,
                                include_count, include_sum, include_mean,
                                include_min, include_max, include_median, include_std
                            )
                            df = _build_stats_df(
                                stats, label,
                                include_count, include_sum, include_mean,
                                include_min, include_max, include_median, include_std
                            )
                            all_dfs.append(df)
                            _save_per_band(df, per_band_output_folder, label)

        if parallel:
            with ThreadPoolExecutor(max_workers=max_workers) as executor:
                list(executor.map(process_path, paths))
        else:
            for p in paths:
                process_path(p)

    if not all_dfs:
        print("⚠️ Warning: No valid data processed, returning empty DataFrame")
        return pd.DataFrame()

    final_df = pd.concat(all_dfs, ignore_index=True)
    final_df = final_df.sort_values(by=["zone_id", "band"], ascending=[True, True])

    if output_csv_path:
        final_df.to_csv(output_csv_path, index=False)
        print(f"✅ Saved combined CSV (sorted by zone_id, band): {output_csv_path}")
    return final_df

def _run_raster_stats_fast(vrt, band, zone_src, tile_size, zone_nod, val_nod,
                           excl, vt,
                           inc_count, inc_sum, inc_mean,
                           inc_min, inc_max, inc_median, inc_std):
    stats = defaultdict(lambda: {"count": 0, "sum": 0.0,
                                 "min": np.inf, "max": -np.inf,
                                 "values": []})
    store_values = inc_median or inc_std

    height, width = zone_src.height, zone_src.width

    for y in range(0, height, tile_size):
        h_tile = min(tile_size, height - y)
        for x in range(0, width, tile_size):
            w = Window(x, y, min(tile_size, width - x), h_tile)

            z = zone_src.read(1, window=w)
            zone_mask = (z != zone_nod) if zone_nod is not None else np.ones_like(z, dtype=bool)

            v = vrt.read(band, window=w)
            mask = zone_mask.copy()
            if val_nod is not None:
                mask &= ~np.isnan(v) if np.isnan(val_nod) else ~np.isclose(v, val_nod)

            # Apply value_transform before exclude_values
            vv = v.astype(np.float32)
            if vt:
                vv = vt(vv)

            # Apply exclude_values after transformation
            for ex in excl:
                mask &= ~ex(vv) if callable(ex) else ~(vv == ex)

            zv = z[mask]
            vv = vv[mask]

            if zv.size == 0:
                continue

            tile_stats = _calc_zonal_tile(zv, vv, store_values)
            _merge_stats(stats, tile_stats,
                         inc_count, inc_sum, inc_min, inc_max, store_values)
    return stats

def _run_array_stats(arr, zone_src, tile_size, excl, vt,
                     inc_count, inc_sum, inc_min,
                     inc_max, inc_median, inc_std,
                     zone_nod):
    height, width = zone_src.height, zone_src.width
    stats = defaultdict(lambda: {"count": 0, "sum": 0.0,
                                 "min": np.inf, "max": -np.inf,
                                 "values": []})
    store_values = inc_median or inc_std

    for y in range(0, height, tile_size):
        h_tile = min(tile_size, height - y)
        for x in range(0, width, tile_size):
            w = Window(x, y, min(tile_size, width - x), h_tile)

            z = zone_src.read(1, window=w)
            zone_mask = (z != zone_nod) if zone_nod is not None else np.ones_like(z, dtype=bool)

            a = arr[y:y + w.height, x:x + w.width]
            if z.shape != a.shape:
                continue

            mask = zone_mask.copy()
            mask &= ~np.isnan(a)

            # Apply value_transform before exclude_values
            aa = a.astype(np.float32)
            if vt:
                aa = vt(aa)

            # Apply exclude_values after transformation
            for ex in excl:
                mask &= ~ex(aa) if callable(ex) else ~(aa == ex)

            zv = z[mask]
            vv = aa[mask]

            if zv.size == 0:
                continue

            tile_stats = _calc_zonal_tile(zv, vv, store_values)
            _merge_stats(stats, tile_stats,
                         inc_count, inc_sum, inc_min, inc_max, store_values)
    return stats

def _calc_zonal_tile(zv, vv, store):
    if zv.size == 0 or vv.size == 0:
        return {}

    ids = np.unique(zv)
    if ids.size == 0:
        return {}

    cnt = np.bincount(zv, minlength=ids.max() + 1)[ids]
    sm = np.bincount(zv, weights=vv, minlength=ids.max() + 1)[ids]
    
    min_vals = np.empty_like(ids, dtype=np.float32)
    max_vals = np.empty_like(ids, dtype=np.float32)
    
    for i, zid in enumerate(ids):
        mask_i = zv == zid
        min_vals[i] = vv[mask_i].min() if mask_i.any() else np.nan
        max_vals[i] = vv[mask_i].max() if mask_i.any() else np.nan

    res = {}
    for i, zid in enumerate(ids):
        res[zid] = {
            "count": int(cnt[i]),
            "sum": float(sm[i]),
            "min": float(min_vals[i]),
            "max": float(max_vals[i]),
            "values": vv[zv == zid].tolist() if store else []
        }
    return res

def _merge_stats(all_s, tile_s, inc_cnt, inc_sum, inc_min, inc_max, store_vals):
    for zid, s in tile_s.items():
        z = all_s[zid]
        if inc_cnt: z["count"] += s["count"]
        if inc_sum: z["sum"] += s["sum"]
        if inc_min and not np.isnan(s["min"]): z["min"] = min(z["min"], s["min"])
        if inc_max and not np.isnan(s["max"]): z["max"] = max(z["max"], s["max"])
        if store_vals and s["values"]:
            z["values"].extend(s["values"])

def _build_stats_df(stats, label,
                    inc_count, inc_sum, inc_mean,
                    inc_min, inc_max, inc_median, inc_std):
    rows = []
    for zid, s in sorted(stats.items()):
        if inc_count and s["count"] == 0:
            continue
        rec = {"zone_id": zid, "band": label}
        if inc_count: rec["count"] = s["count"]
        if inc_sum: rec["sum"] = s["sum"]
        if inc_mean and inc_count and s["count"] > 0: rec["mean"] = s["sum"] / s["count"]
        if inc_min: rec["min"] = s["min"]
        if inc_max: rec["max"] = s["max"]
        if inc_median or inc_std:
            arr = np.array(s["values"])
            if inc_median and arr.size:
                rec["median"] = float(np.median(arr))
            if inc_std and arr.size:
                rec["std"] = float(np.std(arr, ddof=1))
        rows.append(rec)
    df = pd.DataFrame(rows)
    return df.sort_values(by="zone_id", ascending=True)

def _save_per_band(df, folder, label):
    if not folder or df.empty:
        return
    os.makedirs(folder, exist_ok=True)
    path = os.path.join(folder, f"{label}_zonal_stats.csv")
    df = df.sort_values(by="zone_id", ascending=True)
    df.to_csv(path, index=False)
    print(f"✅ Saved per-band CSV: {path}")

We can build an "instance" of the workflow class:

In [ ]:
# Forest height:
zone_raster = r'D:/MaskCanTrans_NA_NALCMS_landcover_2020_30m.tif'
value_raster = r'D:/proj_CanTrans_forest_height_2020.tif'
CanTransStatOut = r'D:/CanTransStatOut'
output_csv = r'D:/CanTransStatOut/CanTransBasin_forest_height_stats.csv'
#transform_values = lambda x: x * 0.1
#transform_values = lambda x: np.tan(np.radians(x)).astype(np.float32)

df = calculate_zonal_stats_streamed(
    zone_raster_path=zone_raster,
    value_raster_path=value_raster,
    bands=None,                  # None means all bands in raster(s)
    tile_size=15360,              # tile size for processing (tune for memory/speed)
    output_csv_path=output_csv,
    per_band_output_folder=CanTransStatOut,
    resampling_method=Resampling.average,
    exclude_values=[0, -9999],     # values to exclude in calculation (e.g., nodata)
    value_transform=None,          # optional function to transform pixel values before stats
    include_median=False,          # median disabled for speed
    include_std=False,             # std disabled for speed
    parallel=True,                 # enable parallel processing per raster file
    max_workers=24                 # number of parallel workers (adjust to CPU cores)
)
print("Zonal statistics completed. Combined dataframe head:")
print(df.head())

In [ ]:
# SLOPE
zone_raster = r'D:/MaskCanTrans_NA_NALCMS_landcover_2020_30m.tif'
value_raster = r'D:/CanTransBasin_merit_hydro_slope.tif'
CanTransStatOut = r'D:/CanTransStatOut'
output_csv = r'D:/CanTransStatOut/CanTransBasin_slope_stats.csv'
transform_values = lambda x: np.tan(np.radians(x)).astype(np.float32)
#
df = calculate_zonal_stats_streamed(
    zone_raster_path=zone_raster,
    value_raster_path=value_raster,     # Glob pattern!
    per_band_output_folder=None,
    output_csv_path=output_csv,         #"zonal_combined.csv"
    resampling_method=Resampling.average,
    exclude_values=[-9999], #, lambda x: x > 90],
    value_transform=transform_values,
    tile_size=15360,
    parallel=True,
    max_workers=24
)
df

In [ ]:
# BDRICM
zone_raster = r'D:/MaskCanTrans_NA_NALCMS_landcover_2020_30m.tif'
value_raster = r'D:/soil/sdep/CanTransBasin_BDRICM_M_250m_ll.tif'
CanTransStatOut = r'D:/CanTransStatOut'
output_csv = r'D:/CanTransStatOut/CanTransBasin_BDRICM_stats.csv'
transform_values = lambda x: x * 0.01

# Run zonal stats calculation:
df = calculate_zonal_stats_streamed(
    zone_raster_path=zone_raster,
    value_raster_path=value_raster,
    bands=None,                            # None means all bands in raster(s)
    tile_size=15360,                       # tile size for processing (tune for memory/speed)
    output_csv_path=output_csv,
    per_band_output_folder=CanTransStatOut,
    resampling_method=Resampling.average,
    exclude_values=[-9999],                 # values to exclude in calculation (e.g., nodata)
    value_transform=transform_values,       # optional function to transform pixel values before stats
    include_median=False,                   # median disabled for speed
    include_std=False,                      # std disabled for speed
    parallel=True,                          # enable parallel processing per raster file
    max_workers=24                          # number of parallel workers (adjust to CPU cores)
)
print("Zonal statistics completed. Combined dataframe head:")
print(df.head())

In [ ]:
# BDTICM
zone_raster = r'D:/MaskCanTrans_NA_NALCMS_landcover_2020_30m.tif'
value_raster = r'D:/soil/sdep/CanTransBasin_BDTICM_M_250m_ll.tif'
CanTransStatOut = r'D:/CanTransStatOut'
output_csv = r'D:/CanTransStatOut/CanTransBasin_BDTICM_stats.csv'
transform_values = lambda x: x * 0.01

# Run zonal stats calculation:
df = calculate_zonal_stats_streamed(
    zone_raster_path=zone_raster,
    value_raster_path=value_raster,
    bands=None,                                     # None means all bands in raster(s)
    tile_size=15360,                                # tile size for processing (tune for memory/speed)
    output_csv_path=output_csv,
    per_band_output_folder=CanTransStatOut,
    resampling_method=Resampling.average,
    exclude_values=[-9999],  #, lambda x: x > 410],  # values to exclude in calculation (e.g., nodata)
    value_transform=transform_values,                # optional function to transform pixel values before stats
    include_median=False,                            # median disabled for speed
    include_std=False,                               # std disabled for speed
    parallel=True,                                   # enable parallel processing per raster file
    max_workers=24                                   # number of parallel workers (adjust to CPU cores)
)
print("Zonal statistics completed. Combined dataframe head:")
print(df.head())

In [ ]:
# BDTICM-Censored
zone_raster = r'D:/MaskCanTrans_NA_NALCMS_landcover_2020_30m.tif'
value_raster = r'D:/soil/sdep/CanTransBasin_CensoredTo4p1m_BDTICM_M_250m_ll.tif'
CanTransStatOut = r'D:/CanTransStatOut'
output_csv = r'D:/CanTransStatOut/CanTransBasin_BDTICM_Censored_stats.csv'
transform_values = lambda x: x * 0.01

# Run zonal stats calculation:
df = calculate_zonal_stats_streamed(
    zone_raster_path=zone_raster,
    value_raster_path=value_raster,
    bands=None,                                     # None means all bands in raster(s)
    tile_size=15360,                                # tile size for processing (tune for memory/speed)
    output_csv_path=output_csv,
    per_band_output_folder=CanTransStatOut,
    resampling_method=Resampling.average,
    exclude_values=[-9999],  #, lambda x: x > 410],  # values to exclude in calculation (e.g., nodata)
    value_transform=transform_values,                # optional function to transform pixel values before stats
    include_median=False,                            # median disabled for speed
    include_std=False,                               # std disabled for speed
    parallel=True,                                   # enable parallel processing per raster file
    max_workers=24                                   # number of parallel workers (adjust to CPU cores)
)
print("Zonal statistics completed. Combined dataframe head:")
print(df.head())

In [ ]:
# CLAY
zone_raster = r'D:/MaskCanTrans_NA_NALCMS_landcover_2020_30m.tif'
value_raster = r'D:/soil/CanTransBasin_CLAY_mesh_weighted.tif'
CanTransStatOut = r'D:/CanTransStatOut'
output_csv = r'D:/CanTransStatOut/CLAY_stats.csv'
# Run zonal stats calculation:
df = calculate_zonal_stats_streamed(
    zone_raster_path=zone_raster,
    value_raster_path=value_raster,
    bands=None,                                   # None means all bands in raster(s)
    tile_size=15360,                              # tile size for processing (tune for memory/speed)
    output_csv_path=output_csv,
    per_band_output_folder=CanTransStatOut,
    resampling_method=Resampling.average,
    exclude_values=[-9999, lambda x: x > 100],    # values to exclude in calculation (e.g., nodata)
    value_transform=None,                         # optional function to transform pixel values before stats
    include_median=False,                         # median disabled for speed
    include_std=False,                            # std disabled for speed
    parallel=True,                                # enable parallel processing per raster file
    max_workers=24                                # number of parallel workers (adjust to CPU cores)
)

In [ ]:
# SAND
zone_raster = r'D:/MaskCanTrans_NA_NALCMS_landcover_2020_30m.tif'
value_raster = r'D:/soil/CanTransBasin_SAND_mesh_weighted.tif'
CanTransStatOut = r'D:/CanTransStatOut'
output_csv = r'D:/CanTransStatOut/SAND_stats.csv'
# Run zonal stats calculation:
df = calculate_zonal_stats_streamed(
    zone_raster_path=zone_raster,
    value_raster_path=value_raster,
    bands=None,                       # None means all bands in raster(s)
    tile_size=15360,                           # tile size for processing (tune for memory/speed)
    output_csv_path=output_csv,
    per_band_output_folder=CanTransStatOut,
    resampling_method=Resampling.average,
    exclude_values=[-9999, lambda x: x > 100],                   # values to exclude in calculation (e.g., nodata)
    value_transform=None,                     # optional function to transform pixel values before stats
    include_median=False,                     # median disabled for speed
    include_std=False,                        # std disabled for speed
    parallel=True,                            # enable parallel processing per raster file
    max_workers=24                            # number of parallel workers (adjust to CPU cores)
)

In [ ]:
# OC and converted into Soil Organic Matter
# Replace values greater than 100.0 with NaN
zone_raster = r'D:/MaskCanTrans_NA_NALCMS_landcover_2020_30m.tif'
value_raster = r'D:/soil/CanTransBasin_OC_mesh_weighted.tif'
CanTransStatOut = r'D:/CanTransStatOut'
output_csv = r'D:/CanTransStatOut/OC_stats.csv'
transform_values = lambda x: x * 0.01 * 1.72
    
# Run zonal stats calculation:
df = calculate_zonal_stats_streamed(
    zone_raster_path=zone_raster,
    value_raster_path=value_raster,
    bands=None,                                    # None means all bands in raster(s)
    tile_size=15360,                               # tile size for processing (tune for memory/speed)
    output_csv_path=output_csv,
    per_band_output_folder=CanTransStatOut,
    resampling_method=Resampling.average,
    exclude_values=[-9999, lambda x: x > 100],     # values to exclude in calculation (e.g., nodata)
    value_transform=transform_values,              # optional function to transform pixel values before stats
    include_median=False,                          # median disabled for speed
    include_std=False,                             # std disabled for speed
    parallel=True,                                 # enable parallel processing per raster file
    max_workers=24                                 # number of parallel workers (adjust to CPU cores)
)

In [ ]:
# LAI
zone_raster = r'D:/MaskCanTrans_NA_NALCMS_landcover_2020_30m.tif'
value_raster = os.path.join(r'D:/monthly_lai_with_water_mask', '*.tif')
CanTransStatOut = r'D:/CanTransStatOut'
output_csv = r'D:/CanTransStatOut/LAI_stats.csv'
transform_values = lambda x: x * 0.1
    
# Run zonal stats calculation:
df = calculate_zonal_stats_streamed(
    zone_raster_path=zone_raster,
    value_raster_path=value_raster,
    bands=None,                                 # None means all bands in raster(s)
    tile_size=15360,                            # tile size for processing (tune for memory/speed)
    output_csv_path=output_csv,
    per_band_output_folder=CanTransStatOut,
    resampling_method=Resampling.average,
    exclude_values=[0, lambda x: x >= 249],     # values to exclude in calculation (e.g., nodata)
    value_transform=transform_values,           # optional function to transform pixel values before stats
    include_median=False,                       # median disabled for speed
    include_std=False,                          # std disabled for speed
    parallel=True,                              # enable parallel processing per raster file
    max_workers=24                              # number of parallel workers (adjust to CPU cores)
)

In [ ]:
# Ksat ratio between at 1m and surface
zone_raster = r'D:/MaskCanTrans_NA_NALCMS_landcover_2020_30m.tif'
value_raster = r'D:/Ksat/k_s_l7_k_s_l1.tif'
CanTransStatOut = r'D:/CanTransStatOut'
output_csv = r'D:/CanTransStatOut/CanTransBasin_KSat_1m_0m_stats.csv'
transform_values = lambda x: x * 1.0

# Run zonal stats calculation:
df = calculate_zonal_stats_streamed(
    zone_raster_path=zone_raster,
    value_raster_path=value_raster,
    bands=None,                                     # None means all bands in raster(s)
    tile_size=15360,                                # tile size for processing (tune for memory/speed)
    output_csv_path=output_csv,
    per_band_output_folder=CanTransStatOut,
    resampling_method=Resampling.average,
    exclude_values=[-9999],  #, lambda x: x > 410],  # values to exclude in calculation (e.g., nodata)
    value_transform=transform_values,                # optional function to transform pixel values before stats
    include_median=False,                            # median disabled for speed
    include_std=False,                               # std disabled for speed
    parallel=True,                                   # enable parallel processing per raster file
    max_workers=24                                   # number of parallel workers (adjust to CPU cores)
)
print("Zonal statistics completed. Combined dataframe head:")
print(df.head())

In [ ]:
# Ksat@ the surface
zone_raster = r'D:/MaskCanTrans_NA_NALCMS_landcover_2020_30m.tif'
value_raster = r'D:/Ksat/k_s_l1.tif'
CanTransStatOut = r'D:/CanTransStatOut'
output_csv = r'D:/CanTransStatOut/CanTransBasin_KSat_0m_stats.csv'
transform_values = lambda x: x * 1

# Run zonal stats calculation:
df = calculate_zonal_stats_streamed(
    zone_raster_path=zone_raster,
    value_raster_path=value_raster,
    bands=None,                                     # None means all bands in raster(s)
    tile_size=15360,                                # tile size for processing (tune for memory/speed)
    output_csv_path=output_csv,
    per_band_output_folder=CanTransStatOut,
    resampling_method=Resampling.average,
    exclude_values=[-9999],  #, lambda x: x > 410],  # values to exclude in calculation (e.g., nodata)
    value_transform=transform_values,                # optional function to transform pixel values before stats
    include_median=False,                            # median disabled for speed
    include_std=False,                               # std disabled for speed
    parallel=True,                                   # enable parallel processing per raster file
    max_workers=24                                   # number of parallel workers (adjust to CPU cores)
)
print("Zonal statistics completed. Combined dataframe head:")
print(df.head())

In [ ]:
# Ksat ratio between at 1m and surface
zone_raster = r'D:/MaskCanTrans_NA_NALCMS_landcover_2020_30m.tif'
value_raster = r'D:/Ksat/k_s_l7_k_s_l1_CensoredToFractionof1.tif'
CanTransStatOut = r'D:/CanTransStatOut'
output_csv = r'D:/CanTransStatOut/CanTransBasin_KSat_1m_0m_stats_censored.csv'
transform_values = lambda x: x * 1

# Run zonal stats calculation:
df = calculate_zonal_stats_streamed(
    zone_raster_path=zone_raster,
    value_raster_path=value_raster,
    bands=None,                                     # None means all bands in raster(s)
    tile_size=15360,                                # tile size for processing (tune for memory/speed)
    output_csv_path=output_csv,
    per_band_output_folder=CanTransStatOut,
    resampling_method=Resampling.average,
    exclude_values=[-9999],  #, lambda x: x > 410],  # values to exclude in calculation (e.g., nodata)
    value_transform=transform_values,                # optional function to transform pixel values before stats
    include_median=False,                            # median disabled for speed
    include_std=False,                               # std disabled for speed
    parallel=True,                                   # enable parallel processing per raster file
    max_workers=24                                   # number of parallel workers (adjust to CPU cores)
)
print("Zonal statistics completed. Combined dataframe head:")
print(df.head())

In [ ]:
# Root Depth
zone_raster = r'D:/MaskCanTrans_NA_NALCMS_landcover_2020_30m.tif'
value_raster = r'D:/RootDepthData/root_depth_censored_0p1_5p0m.tif'
CanTransStatOut = r'D:/CanTransStatOut'
output_csv = r'D:/CanTransStatOut/CanTransBasin_root_depth_stats_censored.csv'
transform_values = lambda x: x * 1

# Run zonal stats calculation:
df = calculate_zonal_stats_streamed(
    zone_raster_path=zone_raster,
    value_raster_path=value_raster,
    bands=None,                                     # None means all bands in raster(s)
    tile_size=15360,                                # tile size for processing (tune for memory/speed)
    output_csv_path=output_csv,
    per_band_output_folder=CanTransStatOut,
    resampling_method=Resampling.average,
    exclude_values=[-9999],  #, lambda x: x > 410],  # values to exclude in calculation (e.g., nodata)
    value_transform=transform_values,                # optional function to transform pixel values before stats
    include_median=False,                            # median disabled for speed
    include_std=False,                               # std disabled for speed
    parallel=True,                                   # enable parallel processing per raster file
    max_workers=24                                   # number of parallel workers (adjust to CPU cores)
)
print("Zonal statistics completed. Combined dataframe head:")
print(df.head())

____